Requirements

In [1]:
import pandas as pd
import numpy as np

1. 	Load Data
Load the dataset directly from API or CSV into Pandas.
 Convert date/time fields (time, updated) to datetime objects.

In [2]:
df=pd.read_csv("../data/earthquakes_raw.csv")
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,us6000d4nn,2020-12-31 23:12:35.050000+00:00,2021-03-06 23:21:08.040000+00:00,-9.1109,118.9156,85.73,4.8,mb,"74 km SSE of Bima, Indonesia",reviewed,...,43.0,NaN,NaN,NaN,NaN,NaN,",dyfi,origin,phase-data,",",us6000d4nn,",",us,",earthquake
1,us6000d4mx,2020-12-31 21:13:14.428000+00:00,2021-03-06 23:21:06.040000+00:00,-22.9389,-66.7445,284.27,4.3,mb,"110 km WSW of Abra Pampa, Argentina",reviewed,...,112.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",us6000d4mx,",",us,",earthquake
2,us6000d4mj,2020-12-31 20:44:20.034000+00:00,2021-03-06 23:21:06.040000+00:00,32.2951,-101.7941,5.00,4.0,mwr,"18 km N of Stanton, Texas",reviewed,...,35.0,NaN,NaN,NaN,NaN,NaN,",dyfi,losspager,moment-tensor,origin,phase-dat...",",us6000d4mj,",",us,",earthquake
3,us6000d7gm,2020-12-31 20:19:33.191000+00:00,2021-03-06 23:21:05.040000+00:00,-14.1143,47.8543,10.00,4.3,mb,"81 km SW of Ambanja, Madagascar",reviewed,...,80.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",us6000d7gm,",",us,",earthquake
4,us6000d4lr,2020-12-31 19:50:17.399000+00:00,2021-03-09 18:33:28.040000+00:00,-0.8052,146.8411,10.00,5.2,mww,"144 km NNW of Lorengau, Papua New Guinea",reviewed,...,34.0,NaN,NaN,NaN,NaN,NaN,",losspager,origin,phase-data,shakemap,",",us6000d4lr,",",us,",earthquake


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81175 entries, 0 to 81174
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              81175 non-null  object 
 1   time            81175 non-null  object 
 2   updated         81175 non-null  object 
 3   latitude        81175 non-null  float64
 4   longitude       81175 non-null  float64
 5   depth_km        81175 non-null  float64
 6   mag             81175 non-null  float64
 7   magType         81175 non-null  object 
 8   place           81175 non-null  object 
 9   status          81175 non-null  object 
 10  tsunami         81175 non-null  int64  
 11  sig             81175 non-null  int64  
 12  net             81175 non-null  object 
 13  nst             57407 non-null  float64
 14  dmin            80720 non-null  float64
 15  rms             81173 non-null  float64
 16  gap             80766 non-null  float64
 17  magError        0 non-null     

In [4]:
df["updated"].dtype
df["time"].dtype

dtype('O')

In [5]:
df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')
df['updated'] = pd.to_datetime(df['updated'], utc=True, errors='coerce')


In [6]:
df["updated"].dtype
df["time"].dtype

datetime64[ns, UTC]

2. 	Clean Text Fields
Use Regex to extract country from place.
 Normalize alert field (if exists) to lowercase.
Ensure all string fields (magType, status, type, net, sources, types) are clean.

In [7]:
df["place"].isnull().sum()

np.int64(0)

In [8]:
def extract_country(place):
  
    split_country=place.split(",")
    return split_country[-1].strip() if len(split_country) > 1 else None

In [9]:
df["country"]=df["place"].apply(extract_country)

In [10]:
string_fields=['magType', 'status', 'type', 'net', 'sources', 'types']

for col in string_fields:
    df[col]=df[col].astype('string').str.strip().str.lower().replace("nan",None)
    

3. 	Clean Numeric Fields

Convert mag, depth_km, nst, dmin, rms, gap, magError, depthError, magNst, sig to numeric.

Fill missing numeric values with 0 or median if needed.


In [11]:
numeric_fields=["mag","depth_km","nst","dmin","rms","magError","depthError","magNst","sig","gap"]

for col in numeric_fields:
    check_null=df[col].isnull().sum()
    print(f"{col}:{check_null}")

mag:0
depth_km:0
nst:23768
dmin:455
rms:2
magError:81175
depthError:81175
magNst:81175
sig:0
gap:409


In [12]:
for col in numeric_fields:
    check_type=df[col].dtype
    print(f"{col}:{check_type}")

mag:float64
depth_km:float64
nst:float64
dmin:float64
rms:float64
magError:float64
depthError:float64
magNst:float64
sig:int64
gap:float64


In [13]:
for col in numeric_fields:
    df[col]=pd.to_numeric(df[col],errors="coerce")

In [14]:
df.loc[:,["nst","dmin","rms","magError","depthError","magNst","gap"]]


,nst,dmin,rms,magError,depthError,magNst,gap
0,NaN,3.316,0.90,NaN,NaN,NaN,43.0
1,NaN,1.322,0.63,NaN,NaN,NaN,112.0
2,NaN,0.316,0.41,NaN,NaN,NaN,35.0
3,NaN,4.477,0.62,NaN,NaN,NaN,80.0
4,NaN,1.337,1.09,NaN,NaN,NaN,34.0
...,...,...,...,...,...,...,...
81170,26.0,2.547,1.33,NaN,NaN,NaN,175.0
81171,12.0,0.686,1.01,NaN,NaN,NaN,121.0
81172,37.0,1.556,1.20,NaN,NaN,NaN,102.0
81173,14.0,2.371,0.74,NaN,NaN,NaN,95.0


In [15]:
null_columns=["nst","dmin","rms","magError","depthError","magNst","gap"]
for col in null_columns:        
        min=df[col].min()
        max=df[col].max()
        mean=df[col].mean()
        median=df[col].median()
        
        print(f"{col}:Mean:{mean}")
        print(f"{col}:Min:{min}")
        print(f"{col}:Max:{max}")
        print(f"{col}:median:{median}")
        print(" ")


nst:Mean:50.164422457191634
nst:Min:0.0
nst:Max:619.0
nst:median:36.0
 
dmin:Mean:3.7838567720976175
dmin:Min:0.0
dmin:Max:62.558
dmin:median:2.316
 
rms:Mean:0.7073491481288394
rms:Min:0.0
rms:Max:2.82
rms:median:0.69
 
magError:Mean:nan
magError:Min:nan
magError:Max:nan
magError:median:nan
 
depthError:Mean:nan
depthError:Min:nan
depthError:Max:nan
depthError:median:nan
 
magNst:Mean:nan
magNst:Min:nan
magNst:Max:nan
magNst:median:nan
 
gap:Mean:106.37874283463769
gap:Min:8.0
gap:Max:348.0
gap:median:103.0
 


c:\Users\roser\OneDrive\Desktop\earthquake_analysis\eqvenv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\roser\OneDrive\Desktop\earthquake_analysis\eqvenv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\roser\OneDrive\Desktop\earthquake_analysis\eqvenv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [16]:
filled_by_mean_value=["nst","dmin","rms","gap"]

df[filled_by_mean_value]=df[filled_by_mean_value].fillna(df[filled_by_mean_value].mean())

In [17]:
filled_by_zero=["magError","depthError","magNst","locationSource","magSource"]

df[filled_by_zero]=df[filled_by_zero].fillna(0)

In [18]:
df.isnull().sum()

id                    0
time                 86
updated               9
latitude              0
longitude             0
depth_km              0
mag                   0
magType               0
place                 0
status                0
tsunami               0
sig                   0
net                   0
nst                   0
dmin                  0
rms                   0
gap                   0
magError              0
depthError            0
magNst                0
locationSource        0
magSource             0
types                 0
ids                   0
sources               0
type                  0
country           18843
dtype: int64

4. 	Add Derived Columns

Year, month, day, day_of_week from time.

Shallow/deep earthquake flag based on depth_km.

Strong/destructive flag based on mag thresholds.

In [19]:
df["year"]=df["time"].dt.year.astype("Int64")
df["month"] = df["time"].dt.month_name().astype("category")
df["day"]=df["time"].dt.day.astype("Int64")
df["day_of_week"]=df["time"].dt.day_name()
df["month"]

0        December
1        December
2        December
3        December
4        December
           ...   
81170    December
81171    December
81172    December
81173    December
81174    December
Name: month, Length: 81175, dtype: category
Categories (12, object): ['April', 'August', 'December', 'February', ..., 'May', 'November', 'October', 'September']

ref : depth category

https://www.usgs.gov/programs/earthquake-hazards/determining-depth-earthquake

In [20]:
df['depth_type'] = np.where(
    (df['depth_km'] <= 70) , 'shallow',
    np.where(
        (df['depth_km'] >= 71) & (df['depth_km'] <= 300), 'intermediate',
        np.where(
            df['depth_km'] > 300, 'deep',
            'other'  
        )
    )
)

ref for mage category : 

https://www.usgs.gov/programs/earthquake-hazards#:~:text=:42%20(UTC)-,Pager%20Alert%20Level:%20Green,Pager%20Alert%20Level:%20Green

In [21]:
df["mag"].min()

np.float64(4.0)

In [22]:
def mag_category(m):
    if 4 <= m < 6:
        return "moderate"
    elif 6 <= m < 7:
        return "strong"
    elif 7 <= m :
        return "destructive"
    else:
        return "N/A"

df["mag_category"] = df["mag"].apply(mag_category)

In [23]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin',
       'rms', 'gap', 'magError', 'depthError', 'magNst', 'locationSource',
       'magSource', 'types', 'ids', 'sources', 'type', 'country', 'year',
       'month', 'day', 'day_of_week', 'depth_type', 'mag_category'],
      dtype='object')

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81175 entries, 0 to 81174
Data columns (total 33 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   id              81175 non-null  object             
 1   time            81089 non-null  datetime64[ns, UTC]
 2   updated         81166 non-null  datetime64[ns, UTC]
 3   latitude        81175 non-null  float64            
 4   longitude       81175 non-null  float64            
 5   depth_km        81175 non-null  float64            
 6   mag             81175 non-null  float64            
 7   magType         81175 non-null  string             
 8   place           81175 non-null  object             
 9   status          81175 non-null  string             
 10  tsunami         81175 non-null  int64              
 11  sig             81175 non-null  int64              
 12  net             81175 non-null  string             
 13  nst             81175 non-null 

In [25]:
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,ids,sources,type,country,year,month,day,day_of_week,depth_type,mag_category
0,us6000d4nn,2020-12-31 23:12:35.050000+00:00,2021-03-06 23:21:08.040000+00:00,-9.1109,118.9156,85.73,4.8,mb,"74 km SSE of Bima, Indonesia",reviewed,...,",us6000d4nn,",",us,",earthquake,Indonesia,2020,December,31,Thursday,intermediate,moderate
1,us6000d4mx,2020-12-31 21:13:14.428000+00:00,2021-03-06 23:21:06.040000+00:00,-22.9389,-66.7445,284.27,4.3,mb,"110 km WSW of Abra Pampa, Argentina",reviewed,...,",us6000d4mx,",",us,",earthquake,Argentina,2020,December,31,Thursday,intermediate,moderate
2,us6000d4mj,2020-12-31 20:44:20.034000+00:00,2021-03-06 23:21:06.040000+00:00,32.2951,-101.7941,5.00,4.0,mwr,"18 km N of Stanton, Texas",reviewed,...,",us6000d4mj,",",us,",earthquake,Texas,2020,December,31,Thursday,shallow,moderate
3,us6000d7gm,2020-12-31 20:19:33.191000+00:00,2021-03-06 23:21:05.040000+00:00,-14.1143,47.8543,10.00,4.3,mb,"81 km SW of Ambanja, Madagascar",reviewed,...,",us6000d7gm,",",us,",earthquake,Madagascar,2020,December,31,Thursday,shallow,moderate
4,us6000d4lr,2020-12-31 19:50:17.399000+00:00,2021-03-09 18:33:28.040000+00:00,-0.8052,146.8411,10.00,5.2,mww,"144 km NNW of Lorengau, Papua New Guinea",reviewed,...,",us6000d4lr,",",us,",earthquake,Papua New Guinea,2020,December,31,Thursday,shallow,moderate


In [ ]:
df.to_csv("../data/cleaned_data.csv",index=False)